# Job Description KSAO 时间序列分析

分析不同职位（onet_code）随时间（季度）对 KSAO 要求的变化。

## 步骤
1. 数据探索和抽样
2. 技能提取
3. 时间序列分析
4. 可视化

## 1. 数据加载和探索

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

# 设置样式
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ 导入完成")

In [ ]:
# ============================================================================
# 配置：设置文件路径
# ============================================================================

# 输入数据
INPUT_FILE = 'academic_postings_unified_individual_rcid_list.parquet'

# 知识库
KB_PATH = '../.skillner-kb/ONET_EN.pkl'

# JD 文本列名（根据你的实际数据调整）
JD_COLUMN = 'job_description'  # 或 'description', 'text' 等

print(f"输入文件: {INPUT_FILE}")
print(f"知识库: {KB_PATH}")

In [ ]:
# 加载数据
print("加载数据...")
df = pd.read_parquet(INPUT_FILE)

print(f"\n总数据量: {len(df):,} 条")
print(f"列: {df.columns.tolist()}")
print(f"\n数据示例:")
df.head()

In [ ]:
# 数据清洗和准备
df['post_date'] = pd.to_datetime(df['post_date'])
df['quarter'] = df['post_date'].dt.to_period('Q')
df['year'] = df['post_date'].dt.year

# 基本统计
print("=== 数据概览 ===")
print(f"不同职位 (onet_code): {df['onet_code'].nunique()}")
print(f"时间范围: {df['post_date'].min()} - {df['post_date'].max()}")
print(f"季度数: {df['quarter'].nunique()}")
print(f"\nTop 10 职位:")
print(df['onet_code'].value_counts().head(10))

## 2. 抽样策略

In [ ]:
# ============================================================================
# 配置：抽样参数
# ============================================================================

# 每个 onet_code × quarter 抽样数量
SAMPLE_SIZE_PER_GROUP = 50  # 根据计算能力调整

# 是否只分析特定职位（可选）
FOCUS_ONET_CODES = None  # 或 ['11-1011.00', '15-1252.00'] 等

print(f"抽样参数:")
print(f"  每组抽样: {SAMPLE_SIZE_PER_GROUP} 条")
print(f"  关注职位: {FOCUS_ONET_CODES or '全部'}")

In [ ]:
# 执行抽样
print("执行分层抽样...")

if FOCUS_ONET_CODES:
    df_filtered = df[df['onet_code'].isin(FOCUS_ONET_CODES)]
else:
    df_filtered = df

sampled_df = df_filtered.groupby(['onet_code', 'quarter']).apply(
    lambda x: x.sample(min(SAMPLE_SIZE_PER_GROUP, len(x)), random_state=42)
).reset_index(drop=True)

print(f"\n原始数据: {len(df_filtered):,} 条")
print(f"抽样后: {len(sampled_df):,} 条 ({len(sampled_df)/len(df_filtered)*100:.2f}%)")
print(f"\nonet_code × quarter 组合数: {sampled_df.groupby(['onet_code', 'quarter']).ngroups}")

# 保存抽样数据
sampled_df.to_parquet('sampled_jd_data.parquet', index=False)
print("\n✓ 抽样数据已保存到 sampled_jd_data.parquet")

## 3. 技能提取

### 方法选择

- **Fuzzy Matching** (推荐)：快速，适合大规模处理
- **Semantic Matching**：慢但准确，需要安装 sentence-transformers

In [ ]:
import sys
sys.path.insert(0, '..')

from skillner.onet_converter import load_kb
from skillner.enhanced_matching import FuzzyQueryMethod
from skillner.core import Pipeline, Document
from skillner.text_loaders import StrTextLoader
from skillner.matchers import SlidingWindowMatcher
from skillner.conflict_resolvers import SpanProcessor

# 加载知识库
print("加载知识库...")
kb = load_kb(KB_PATH)
print(f"✓ 知识库: {len(kb)} 个技能")

# 创建查询方法（使用 Fuzzy Matching）
print("\n创建查询方法...")
query_method = FuzzyQueryMethod(kb, similarity_threshold=0.85)
print("✓ 使用 Fuzzy Matching (快速)")

# 如果要使用语义匹配，取消下面的注释：
# from skillner.enhanced_matching import SemanticQueryMethod
# query_method = SemanticQueryMethod(kb, similarity_threshold=0.6)
# print("✓ 使用 Semantic Matching (准确但慢)")

In [ ]:
# 定义提取函数
def extract_skills_from_jd(job_description, query_method):
    """从 job description 提取技能"""
    doc = Document()
    pipeline = Pipeline()
    
    pipeline.add_node(StrTextLoader(job_description), name='loader')
    pipeline.add_node(
        SlidingWindowMatcher(
            query_method,
            max_window_size=5,
            pre_filter=lambda w: w.lower()
        ),
        name='matcher'
    )
    pipeline.add_node(
        SpanProcessor(
            dict_filters={'max_candidate': lambda span: max(span.li_candidates, key=len)}
        ),
        name='resolver'
    )
    
    pipeline.run(doc)
    
    # 收集技能
    skills = []
    for sentence in doc:
        for span in sentence.li_spans:
            candidate = span.metadata.get('max_candidate')
            if candidate:
                skills.append({
                    'skill': candidate.metadata['pref_label'],
                    'section': candidate.metadata.get('section', 'Unknown')
                })
    
    return skills

print("✓ 提取函数已定义")

In [ ]:
# 测试单条提取
print("测试提取...\n")

sample_jd = sampled_df.iloc[0][JD_COLUMN]
print(f"JD 示例 (前300字):")
print(sample_jd[:300] + "...\n")

import time
start = time.time()
test_skills = extract_skills_from_jd(sample_jd, query_method)
elapsed = time.time() - start

print(f"提取结果: {len(test_skills)} 个技能，耗时 {elapsed:.2f}s")
print(f"\n技能示例:")
for skill in test_skills[:10]:
    print(f"  - {skill['skill']} ({skill['section']})")

print(f"\n估算全量处理时间: {elapsed * len(sampled_df) / 3600:.1f} 小时")

In [ ]:
# 批量提取（带进度条）
print("开始批量提取...\n")

results = []

for idx, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="提取技能"):
    try:
        jd_text = row[JD_COLUMN]
        skills = extract_skills_from_jd(jd_text, query_method)
        
        # 按 section 分组
        skills_by_section = {}
        for skill in skills:
            section = skill['section']
            if section not in skills_by_section:
                skills_by_section[section] = []
            skills_by_section[section].append(skill['skill'])
        
        results.append({
            'index': idx,
            'onet_code': row['onet_code'],
            'post_date': row['post_date'],
            'quarter': str(row['quarter']),
            'year': row['year'],
            'num_skills_total': len(skills),
            'num_skills_unique': len(set(s['skill'] for s in skills)),
            'skills': [s['skill'] for s in skills],
            'skills_by_section': skills_by_section,
            # KSAO 分类
            'knowledge': skills_by_section.get('Knowledge', []),
            'skills': skills_by_section.get('Skills', []),
            'abilities': skills_by_section.get('Abilities', []),
            'technology': skills_by_section.get('Technology Skills', [])
        })
    except Exception as e:
        print(f"\nError processing row {idx}: {e}")
        continue

# 转为 DataFrame
results_df = pd.DataFrame(results)

print(f"\n✓ 完成！处理了 {len(results_df)} 条记录")
print(f"平均每条提取: {results_df['num_skills_total'].mean():.1f} 个技能")

# 保存结果
results_df.to_parquet('jd_skills_extracted.parquet', index=False)
print("\n✓ 结果已保存到 jd_skills_extracted.parquet")

## 4. 时间序列分析

In [ ]:
# 加载提取结果（如果之前已运行过）
# results_df = pd.read_parquet('jd_skills_extracted.parquet')

# 查看数据
print("提取结果概览:")
results_df.head()

In [ ]:
# 按职位和季度聚合
temporal_stats = results_df.groupby(['onet_code', 'quarter']).agg({
    'num_skills_total': 'mean',
    'num_skills_unique': 'mean',
    'index': 'count'  # JD 数量
}).reset_index()

temporal_stats.columns = ['onet_code', 'quarter', 'avg_skills_total', 'avg_skills_unique', 'num_jds']

print("时间序列数据:")
temporal_stats.head(10)

In [ ]:
# 选择几个代表性职位进行可视化
top_onet_codes = results_df['onet_code'].value_counts().head(5).index.tolist()

print(f"Top 5 职位: {top_onet_codes}")

# 绘图
fig, axes = plt.subplots(len(top_onet_codes), 1, figsize=(14, 4*len(top_onet_codes)))

if len(top_onet_codes) == 1:
    axes = [axes]

for idx, onet_code in enumerate(top_onet_codes):
    data = temporal_stats[temporal_stats['onet_code'] == onet_code].copy()
    data = data.sort_values('quarter')
    
    ax = axes[idx]
    ax.plot(range(len(data)), data['avg_skills_total'], marker='o', label='Total Skills')
    ax.plot(range(len(data)), data['avg_skills_unique'], marker='s', label='Unique Skills')
    
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels([str(q) for q in data['quarter']], rotation=45)
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Average Skills per JD')
    ax.set_title(f'ONET Code: {onet_code}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('temporal_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ 图表已保存到 temporal_trends.png")

In [ ]:
# 分析 KSAO 类别的时间变化
# 展开 skills_by_section
ksao_data = []

for _, row in results_df.iterrows():
    for section, skills in row['skills_by_section'].items():
        ksao_data.append({
            'onet_code': row['onet_code'],
            'quarter': row['quarter'],
            'section': section,
            'num_skills': len(skills)
        })

ksao_df = pd.DataFrame(ksao_data)

# 按职位、季度、section 聚合
ksao_temporal = ksao_df.groupby(['onet_code', 'quarter', 'section']).agg({
    'num_skills': 'mean'
}).reset_index()

print("KSAO 时间序列数据:")
ksao_temporal.head(10)

In [ ]:
# 绘制 KSAO 类别趋势（堆叠面积图）
onet_code = top_onet_codes[0]  # 选择第一个职位

data = ksao_temporal[ksao_temporal['onet_code'] == onet_code]
pivot = data.pivot(index='quarter', columns='section', values='num_skills').fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot.area(ax=ax, alpha=0.7)

ax.set_xlabel('Quarter')
ax.set_ylabel('Average Number of Skills')
ax.set_title(f'KSAO Trends Over Time - {onet_code}')
ax.legend(title='Section', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('ksao_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ 图表已保存到 ksao_trends.png")

## 5. 具体技能变化分析

In [ ]:
# 展开所有技能
skill_records = []

for _, row in results_df.iterrows():
    for section, skills in row['skills_by_section'].items():
        for skill in skills:
            skill_records.append({
                'onet_code': row['onet_code'],
                'quarter': row['quarter'],
                'section': section,
                'skill': skill
            })

skill_detail_df = pd.DataFrame(skill_records)

print(f"总记录数: {len(skill_detail_df):,}")
skill_detail_df.head()

In [ ]:
# 分析特定职位的技能变化
onet_code = top_onet_codes[0]

# 计算每个技能在每个季度的出现频率
skill_freq = skill_detail_df[skill_detail_df['onet_code'] == onet_code].groupby(
    ['quarter', 'skill']
).size().reset_index(name='frequency')

# 找出最常见的技能
top_skills = skill_freq.groupby('skill')['frequency'].sum().sort_values(ascending=False).head(10).index

# 绘制这些技能的时间趋势
fig, ax = plt.subplots(figsize=(14, 6))

for skill in top_skills:
    data = skill_freq[skill_freq['skill'] == skill].sort_values('quarter')
    ax.plot(range(len(data)), data['frequency'], marker='o', label=skill)

ax.set_xticks(range(len(data)))
ax.set_xticklabels([str(q) for q in data['quarter']], rotation=45)
ax.set_xlabel('Quarter')
ax.set_ylabel('Frequency')
ax.set_title(f'Top 10 Skills Trends - {onet_code}')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('top_skills_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ 图表已保存到 top_skills_trends.png")

In [ ]:
# 识别新兴技能和衰退技能
quarters = sorted(skill_freq['quarter'].unique())
first_quarter = quarters[0]
last_quarter = quarters[-1]

# 第一季度和最后一季度的技能频率
first_q_skills = skill_freq[skill_freq['quarter'] == first_quarter].set_index('skill')['frequency']
last_q_skills = skill_freq[skill_freq['quarter'] == last_quarter].set_index('skill')['frequency']

# 计算变化
all_skills = set(first_q_skills.index) | set(last_q_skills.index)

skill_changes = []
for skill in all_skills:
    first_freq = first_q_skills.get(skill, 0)
    last_freq = last_q_skills.get(skill, 0)
    change = last_freq - first_freq
    
    if first_freq > 0:
        pct_change = (change / first_freq) * 100
    else:
        pct_change = float('inf') if last_freq > 0 else 0
    
    skill_changes.append({
        'skill': skill,
        'first_quarter_freq': first_freq,
        'last_quarter_freq': last_freq,
        'absolute_change': change,
        'pct_change': pct_change
    })

changes_df = pd.DataFrame(skill_changes)

# 新兴技能（增长最快）
print(f"\n=== 新兴技能 ({first_quarter} → {last_quarter}) ===")
emerging = changes_df[changes_df['pct_change'] != float('inf')].nlargest(10, 'pct_change')
print(emerging[['skill', 'first_quarter_freq', 'last_quarter_freq', 'pct_change']])

# 衰退技能（下降最快）
print(f"\n=== 衰退技能 ({first_quarter} → {last_quarter}) ===")
declining = changes_df[changes_df['pct_change'] != float('inf')].nsmallest(10, 'pct_change')
print(declining[['skill', 'first_quarter_freq', 'last_quarter_freq', 'pct_change']])

## 6. 导出结果

In [ ]:
# 导出汇总统计
summary = {
    'temporal_stats': temporal_stats,
    'ksao_temporal': ksao_temporal,
    'skill_changes': changes_df
}

for name, data in summary.items():
    filename = f'{name}.csv'
    data.to_csv(filename, index=False)
    print(f"✓ 已保存: {filename}")

print("\n所有分析结果已导出！")